###### Imports and Settings

In [1]:
import pandas as pd
import numpy as np
import requests
#import io
import pickle
from collections import deque
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 150)
pd.options.mode.chained_assignment = None  # default='warn'

# Comprehensive Plan Data Pull  

In [2]:
#to read in... rb is read bite
with open('api_keys.pkl', 'rb') as keys_file:
        keys_dict_2 = pickle.load(keys_file)

In [3]:
#create a variable that contains your api key
api_key = keys_dict_2['CENSUS']

In [4]:
GNRC = ['161', #Stewart
       '125', #Montgomery
       '083', #Houston
       '085', #Humphreys
       '043', #Dickson
       '021', #Cheatham
       '147', #Robertson
       '165', #Sumner
       '037', #Davidson
       '189', #Wilson
       '169', #Trousdale
       '187', #Williamson
       '149', #Rutherford
       '119'] #Maury

In [5]:
qlaces = ['1600000US4702180', #Ashland City: Cheatham
          '1600000US4739660', #Kingston Springs: Cheatham
          '1600000US4757480', #Pegram: Cheatham
          '1600000US4759560', #Pleasant View: Cheatham
          '1600000US4704620', #Belle Meade: Davidson
          '1600000US4705140', #Berry Hill: Davidson
          '1600000US4727020', #Forest Hills: Davidson
          '1600000US4729920', #Goodlettsville: Davidson/Sumner
          '1600000US4752006', #Nashville-Davidson metropolitan government (balance): Davidson
          '1600000US4754780', #Oak Hill: Davidson
          '1600000US4763140', #Ridgetop: Davidson/Robertson
          '1600000US4709880', #Burns: Dickson
          '1600000US4713080', #Charlotte: Dickson
          '1600000US4720620', #Dickson: Dickson
          '1600000US4769080', #Slayden: Dickson
          '1600000US4776860', #Vanleer: Dickson
          '1600000US4779980', #White Bluff: Dickson
          '1600000US4724320', #Erin: Houston
          '1600000US4773460', #Tennessee Ridge: Houston/Stewart
          '1600000US4744840', #McEwen: Humphreys
          '1600000US4752820', #New Johnsonville: Humphreys
          '1600000US4778560', #Waverly: Humphreys
          '1600000US4716540', #Columbia: Maury
          '1600000US4751080', #Mount Pleasant: Maury
          '1600000US4770580', #Spring Hill: Maury/Williamson
          '1600000US4715160', #Clarksville: Montgomery
          '1600000US4700200', #Adams: Robertson
          '1600000US4711980', #Cedar Hill: Robertson
          '1600000US4716980', #Coopertown: Robertson
          '1600000US4718420', #Cross Plains: Robertson
          '1600000US4730960', #Greenbrier: Robertson
          '1600000US4748980', #Millersville: Robertson/Sumner
          '1600000US4760280', #Portland: Robertson/Sumner
          '1600000US4770500', #Springfield: Robertson
          '1600000US4780200', #White House: Robertson/Sumner
          '1600000US4722360', #Eagleville: Rutherford
          '1600000US4741200', #La Vergne: Rutherford
          '1600000US4751560', #Murfreesboro: Rutherford
          '1600000US4769420', #Smyrna: Rutherford
          '1600000US4718820', #Cumberland City: Stewart
          '1600000US4721400', #Dover: Stewart
          '1600000US4728540', #Gallatin: Sumner
          '1600000US4733280', #Hendersonville: Sumner
          '1600000US4779420', #Westmoreland: Sumner
          '1600000US4708280', #Brentwood: Williamson
          '1600000US4725440', #Fairview: Williamson
          '1600000US4727740', #Franklin: Williamson
          '1600000US4753460', #Nolensville: Williamson
          '1600000US4773900', #Thompson's Station: Williamson
          '1600000US4741520', #Lebanon: Wilson
          '1600000US4750780', #Mount Juliet: Wilson
          '1600000US4778320'] #Watertown: Wilson

## Read In Data Guide

The "head" should never be more than 2 variables, and the "tail" never more than 2. Since we can pull 50 variables at once this means that we can systematically program in 46 variables for each pull, so that's:
+ dg1: ID's 1 - 46  
+ dg2: ID's 47-92  
**This data guide stops at ID 88 as of 7/8/2022.**

### SF1

In [6]:
dataguide = pd.read_csv('../../data guides/DATA GUIDE SF12000.csv', dtype = str)
dataguide['ID'] = dataguide['ID'].astype(int)

In [7]:
dg1 = dataguide[dataguide['ID'].between(1, 46)]
dg2 = dataguide[dataguide['ID'].between(47, 92)]

In [8]:
# ONE 2000
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg1['SF1 Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg1['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
url_str= 'https://api.census.gov/data/2000/dec/sf1?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "county:*"
predicates["in"]= "state:47"                                                             
data = requests.get(url_str, params= predicates)                                                                
col_names = col_list
df = pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
df = df.loc[df['GeoFIPS'].isin(GNRC)]
#places call
url_str= 'https://api.census.gov/data/2000/dec/sf1?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[places['GEO_ID'].isin(qlaces)]
df = df.append(places).reset_index(drop = True)
# #tract call
# url_str= 'https://api.census.gov/data/2000/dec/sf1?key='+api_key
# predicates= {}
# get_vars= var_list
# predicates['get'] = ','.join(get_vars)
# predicates['for'] = 'tract:0902'
# predicates['in'] = 'state:47'
# predicates['%20'] = 'county:169'
# data= requests.get(url_str, params= predicates)
# col_names = col_list
# tract=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
# df = df.append(tract).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2000/dec/sf1?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
df = df.append(state, ignore_index = True)
#onepull = df
#national call
url_str= 'https://api.census.gov/data/2000/dec/sf1?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "us:*"
data= requests.get(url_str, params= predicates)
col_names = col_list
national=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
national['GeoFIPS'] = '0'
df = df.append(national, ignore_index = True)
onepull = df
print('Okay Finished')

Okay Finished


In [9]:
one = onepull

In [10]:
one.tail()

,NAME,GEO_ID,pop,agebysex_total_series,age_total_male,age_m_u5,age_m_5to9,age_m_10to14,age_m_15to17,age_m_18to19,age_m_20,age_m_21,age_m_22to24,age_m_25to29,age_m_30to34,age_m_35to39,age_m_40to44,age_m_45to49,age_m_50to54,age_m_55to59,age_m_60to61,age_m_62to64,age_m_65to66,age_m_67to69,age_m_70to74,age_m_75to79,age_m_80to84,age_m_85+,age_total_female,age_f_u5,age_f_5to9,age_f_10to14,age_f_15to17,age_f_18to19,age_f_20,age_f_21,age_f_22to24,age_f_25to29,age_f_30to34,age_f_35to39,age_f_40to44,age_f_45to49,age_f_50to54,age_f_55to59,age_f_60to61,age_f_62to64,age_f_65to66,age_f_67to69,StateFIPS,GeoFIPS
63,"Westmoreland town, Tennessee",1600000US4779420,2093,2093,934,83,69,61,39,25,13,13,34,73,71,67,60,55,70,51,21,21,14,24,21,28,9,12,1159,69,77,86,46,31,6,13,38,74,71,82,66,80,75,55,16,26,18,34,47,79420
64,"White Bluff town, Tennessee",1600000US4779980,2142,2142,1076,64,103,80,49,17,9,15,46,86,95,74,75,85,71,53,22,29,19,20,31,23,5,5,1066,64,69,61,33,29,6,10,44,98,68,66,95,68,81,65,18,29,18,29,47,79980
65,"White House city, Tennessee",1600000US4780200,7220,7220,3603,365,346,336,169,87,35,31,84,275,363,410,311,205,187,128,34,59,27,44,41,40,18,8,3617,271,362,299,140,67,40,34,102,271,372,432,292,221,192,128,55,64,32,51,47,80200
66,Tennessee,0400000US47,5689283,5689283,2770275,192283,203330,202772,120149,82409,41238,39546,113958,203113,205955,223109,219802,201425,183503,142367,47954,65480,39196,54065,77213,56970,32595,21843,2919008,182597,192483,192383,112524,80102,40563,39361,111679,200716,206117,230218,229398,211279,190709,151575,52525,73350,45977,65333,47,0
67,United States,0100000US,281421906,281421906,138053563,9810733,10523277,10520197,6204989,4186015,2071220,1965673,5650921,9798760,10321769,11318696,11129102,9889506,8607724,6508729,2173239,2963388,1814807,2585555,3902912,3044456,1834897,1226998,143368343,9365065,10026228,10007875,5835448,3993438,1978228,1875409,5422550,9582576,10188619,11387968,11312761,10202898,8977824,6960508,2367932,3300888,2075424,3057759,1,0


In [11]:
# url_str = 'https://api.census.gov/data/2000/dec/sf1?'
# predicates = {}
# get_vars = ['NAME', 'GEO_ID', 'P001001']
# predicates['get'] = ','.join(get_vars)
# predicates['for'] = 'tract:0902'
# predicates['in'] = 'state:47'
# predicates['%20'] = 'county:169'
# data = requests.get(url_str)
# col_names = ['NAME', 'GEOID','Variable','State','County']
# data = pd.DataFrame(data=data.json()[1:], columns=col_names)
# data.head()

In [12]:
# url_str = 'https://api.census.gov/data/2000/dec/sf1?get=P001001,NAME&for=tract:0902&in=state:47%20county:169'
# predicates = {}
# get_vars= var_list
# predicates['get'] = ','.join(get_vars)
# predicates['for'] = 'tract:0902'
# predicates['in'] = 'state:47'
# predicates['%20'] = 'county:169'
# data = requests.get(url_str)
# col_names = col_list
# tract=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
# data.head()

In [13]:
# TWO 2000
# listparams
head1 = 'NAME'
head2 = 'GEO_ID'
tail_cols1 = 'StateFIPS'
tail_cols2 = 'GeoFIPS'
#make variables list
var_list = list(dg2['SF1 Variable'])
#add stuff to variables list
var_list = deque(var_list)
var_list.appendleft(head2)
var_list.appendleft(head1)
var_list = list(var_list)
#make columns list
col_list = list(dg2['Column Name'])
#add stuff to columns list
col_list.append(tail_cols1)
col_list.append(tail_cols2)
col_list = deque(col_list)
col_list.appendleft(head2)
col_list.appendleft(head1)
col_list = list(col_list)
#add stuff to columns list
url_str= 'https://api.census.gov/data/2000/dec/sf1?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "county:*"
predicates["in"]= "state:47"                                                             
data = requests.get(url_str, params= predicates)                                                                
col_names = col_list
df = pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
df = df.loc[df['GeoFIPS'].isin(GNRC)]
#places call
url_str= 'https://api.census.gov/data/2000/dec/sf1?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "place:*"
predicates["in"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
places=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
places=places.loc[places['GEO_ID'].isin(qlaces)]
df = df.append(places).reset_index(drop = True)
#state call
col_list.remove(tail_cols2)
url_str= 'https://api.census.gov/data/2000/dec/sf1?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "state:47"
data= requests.get(url_str, params= predicates)
col_names = col_list
state=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
state['GeoFIPS'] = '0'
df = df.append(state, ignore_index = True)
#national call
url_str= 'https://api.census.gov/data/2000/dec/sf1?key='+api_key
predicates= {}
get_vars= var_list
predicates["get"]= ",". join(get_vars)
predicates["for"]= "us:*"
data= requests.get(url_str, params= predicates)
col_names = col_list
national=pd.DataFrame(columns=col_names, data=data.json()[1:], dtype=str)
national['GeoFIPS'] = '0'
df = df.append(national, ignore_index = True)
twopull = df
print('Okay Finished')

Okay Finished


In [14]:
two = twopull

In [15]:
last = two

## Joining

In [16]:
one = one.drop(columns = ['StateFIPS','GeoFIPS'])
last = last.drop(columns = 'NAME')

In [17]:
data = one.merge(last, on = 'GEO_ID')

In [18]:
transp = data.transpose()
transp.columns = transp.iloc[0]
transp.head()

NAME,"Cheatham County, Tennessee","Davidson County, Tennessee","Dickson County, Tennessee","Houston County, Tennessee","Humphreys County, Tennessee","Maury County, Tennessee","Montgomery County, Tennessee","Robertson County, Tennessee","Rutherford County, Tennessee","Stewart County, Tennessee","Sumner County, Tennessee","Trousdale County, Tennessee","Williamson County, Tennessee","Wilson County, Tennessee","Adams city, Tennessee","Ashland City town, Tennessee","Belle Meade city, Tennessee","Berry Hill city, Tennessee","Brentwood city, Tennessee","Burns town, Tennessee","Cedar Hill city, Tennessee","Charlotte town, Tennessee","Clarksville city, Tennessee","Columbia city, Tennessee","Coopertown town, Tennessee","Cross Plains city, Tennessee","Cumberland City town, Tennessee","Dickson city, Tennessee","Dover city, Tennessee","Eagleville city, Tennessee","Erin city, Tennessee","Fairview city, Tennessee","Forest Hills city, Tennessee","Franklin city, Tennessee","Gallatin city, Tennessee","Goodlettsville city, Tennessee","Greenbrier town, Tennessee","Hendersonville city, Tennessee","Kingston Springs town, Tennessee","La Vergne city, Tennessee","Lebanon city, Tennessee","McEwen city, Tennessee","Millersville city, Tennessee","Mount Juliet city, Tennessee","Mount Pleasant city, Tennessee","Murfreesboro city, Tennessee","Nashville-Davidson (balance), Tennessee","New Johnsonville city, Tennessee","Nolensville town, Tennessee","Oak Hill city, Tennessee","Pegram town, Tennessee","Pleasant View city, Tennessee","Portland city, Tennessee","Ridgetop city, Tennessee","Slayden town, Tennessee","Smyrna town, Tennessee","Springfield city, Tennessee","Spring Hill city, Tennessee","Tennessee Ridge town, Tennessee","Thompson's Station town, Tennessee","Vanleer town, Tennessee","Watertown city, Tennessee","Waverly city, Tennessee","Westmoreland town, Tennessee","White Bluff town, Tennessee","White House city, Tennessee",Tennessee,United States
NAME,"Cheatham County, Tennessee","Davidson County, Tennessee","Dickson County, Tennessee","Houston County, Tennessee","Humphreys County, Tennessee","Maury County, Tennessee","Montgomery County, Tennessee","Robertson County, Tennessee","Rutherford County, Tennessee","Stewart County, Tennessee","Sumner County, Tennessee","Trousdale County, Tennessee","Williamson County, Tennessee","Wilson County, Tennessee","Adams city, Tennessee","Ashland City town, Tennessee","Belle Meade city, Tennessee","Berry Hill city, Tennessee","Brentwood city, Tennessee","Burns town, Tennessee","Cedar Hill city, Tennessee","Charlotte town, Tennessee","Clarksville city, Tennessee","Columbia city, Tennessee","Coopertown town, Tennessee","Cross Plains city, Tennessee","Cumberland City town, Tennessee","Dickson city, Tennessee","Dover city, Tennessee","Eagleville city, Tennessee","Erin city, Tennessee","Fairview city, Tennessee","Forest Hills city, Tennessee","Franklin city, Tennessee","Gallatin city, Tennessee","Goodlettsville city, Tennessee","Greenbrier town, Tennessee","Hendersonville city, Tennessee","Kingston Springs town, Tennessee","La Vergne city, Tennessee","Lebanon city, Tennessee","McEwen city, Tennessee","Millersville city, Tennessee","Mount Juliet city, Tennessee","Mount Pleasant city, Tennessee","Murfreesboro city, Tennessee","Nashville-Davidson (balance), Tennessee","New Johnsonville city, Tennessee","Nolensville town, Tennessee","Oak Hill city, Tennessee","Pegram town, Tennessee","Pleasant View city, Tennessee","Portland city, Tennessee","Ridgetop city, Tennessee","Slayden town, Tennessee","Smyrna town, Tennessee","Springfield city, Tennessee","Spring Hill city, Tennessee","Tennessee Ridge town, Tennessee","Thompson's Station town, Tennessee","Vanleer town, Tennessee","Watertown city, Tennessee","Waverly city, Tennessee","Westmoreland town, Tennessee","White Bluff town, Tennessee","White House city, Tennessee",Tennessee,United States
GEO_ID,0500000US47021,0500000US47037,0500000US47043,0500000US47083,0500000US47085,0500000U

In [19]:
transp = transp.loc[transp['Stewart County, Tennessee'] != 'Stewart County, Tennessee']
transp = transp.loc[transp['Stewart County, Tennessee'] != '0500000US47161']

In [20]:
numcols = list(transp.columns)
numcols
transp[numcols] = transp[numcols].astype(float)

In [21]:
data = transp

In [22]:
GNRCCounties = [data['Stewart County, Tennessee'],data['Montgomery County, Tennessee'],data['Houston County, Tennessee'],data['Humphreys County, Tennessee'],
                data['Dickson County, Tennessee'],data['Cheatham County, Tennessee'],data['Robertson County, Tennessee'],data['Sumner County, Tennessee'],
                data['Davidson County, Tennessee'],data['Wilson County, Tennessee'],data['Trousdale County, Tennessee'],data['Williamson County, Tennessee'],
                data['Rutherford County, Tennessee']]
data['GNRC'] = sum(GNRCCounties)

In [23]:
GNRCCountiesAll = [data['Stewart County, Tennessee'],data['Montgomery County, Tennessee'],data['Houston County, Tennessee'],data['Humphreys County, Tennessee'],
                data['Dickson County, Tennessee'],data['Cheatham County, Tennessee'],data['Robertson County, Tennessee'],data['Sumner County, Tennessee'],
                data['Davidson County, Tennessee'],data['Wilson County, Tennessee'],data['Trousdale County, Tennessee'],data['Williamson County, Tennessee'],
                data['Rutherford County, Tennessee'],data['Maury County, Tennessee']]
data['GNRC Region'] = sum(GNRCCountiesAll)

In [24]:
MPOCounties = [data['Robertson County, Tennessee'],data['Sumner County, Tennessee'],data['Davidson County, Tennessee'],data['Wilson County, Tennessee'],
               data['Williamson County, Tennessee'],data['Rutherford County, Tennessee'],data['Maury County, Tennessee']]
data['MPO'] = sum(MPOCounties)

In [25]:
RuthInc = [data['Eagleville city, Tennessee'],data['La Vergne city, Tennessee'],data['Murfreesboro city, Tennessee'],data['Smyrna town, Tennessee']]
data['Rutherford Incorporated'] = sum(RuthInc)
data['Rutherford Unincorporated'] = data['Rutherford County, Tennessee'] - data['Rutherford Incorporated']
WilsonInc = [data['Lebanon city, Tennessee'],data['Mount Juliet city, Tennessee'],data['Watertown city, Tennessee']]
data['Wilson Incorporated'] = sum(WilsonInc)
data['Wilson Unincorporated'] = data['Wilson County, Tennessee'] - data['Wilson Incorporated']
CheathInc = [data['Ashland City town, Tennessee'],data['Kingston Springs town, Tennessee'],data['Pegram town, Tennessee'],data['Pleasant View city, Tennessee']]
data['Cheatham Incorporated'] = sum(CheathInc)
data['Cheatham Unincorporated'] = data['Cheatham County, Tennessee'] - data['Cheatham Incorporated']
DicksInc = [data['Burns town, Tennessee'],data['Charlotte town, Tennessee'],data['Dickson city, Tennessee'],data['Slayden town, Tennessee'],
            data['Vanleer town, Tennessee'],data['White Bluff town, Tennessee']]
data['Dickson Incorporated'] = sum(DicksInc)
data['Dickson Unincorporated'] = data['Dickson County, Tennessee'] - data['Dickson Incorporated']
HumphInc = [data['McEwen city, Tennessee'],data['New Johnsonville city, Tennessee'],data['Waverly city, Tennessee']]
data['Humphreys Incorporated'] = sum(HumphInc)
data['Humphreys Unincorporated'] = data['Humphreys County, Tennessee'] - data['Humphreys Incorporated']
data['Montgomery Incorporated'] = data['Clarksville city, Tennessee']
data['Montgomery Unincorporated'] = data['Montgomery County, Tennessee'] - data['Montgomery Incorporated']

In [26]:
data = data.transpose().reset_index()

In [27]:
data.head()

,NAME,pop,agebysex_total_series,age_total_male,age_m_u5,age_m_5to9,age_m_10to14,age_m_15to17,age_m_18to19,age_m_20,age_m_21,age_m_22to24,age_m_25to29,age_m_30to34,age_m_35to39,age_m_40to44,age_m_45to49,age_m_50to54,age_m_55to59,age_m_60to61,age_m_62to64,age_m_65to66,age_m_67to69,age_m_70to74,age_m_75to79,age_m_80to84,age_m_85+,age_total_female,age_f_u5,age_f_5to9,age_f_10to14,age_f_15to17,age_f_18to19,age_f_20,age_f_21,age_f_22to24,age_f_25to29,age_f_30to34,age_f_35to39,age_f_40to44,age_f_45to49,age_f_50to54,age_f_55to59,age_f_60to61,age_f_62to64,age_f_65to66,age_f_67to69,age_f_70to74,age_f_75to79,age_f_80to84,age_f_85+,raceeth_total_series,raceeth_total_oneracealone,raceeth_white_alone,raceeth_blackafricanamerican_alone,raceeth_americanindianalaskanative_alone,raceeth_asian_alone,raceeth_nativehawaiianotherpacificislander_alone,raceeth_someotherrace_alone,raceeth_twoormoreraces_alone,raceeth_whitealone_nothispanicorlatino,raceeth_hispanicorlatino,hhsize_avg,units_allhousing,occupancy_total_series,occupancy_occupiedunits,occupancy_vacantunits,tenure_total_series,tenure_owneroccunits,tenure_renteroccunits,hhtype_total_series,hhtype_oneperson,hhtype_oneperson_male,hhtype_oneperson_female,hhtype_twoormore,hhtype_twoormore_family,hhtype_twoormore_family_marriedcouplefam,hhtype_twoormore_family_marriedcouplefam_ownchildrenunder18,hhtype_twoormore_family_marriedcouplefam_noownchildrenunder18,hhtype_twoormore_family_otherfam,hhtype_twoormore_family_otherfam_malenospouse,hhtype_twoormore_family_otherfam_malenospouse_ownchildrenunder18,hhtype_twoormore_family_otherfam_malenospouse_noownchildrenunder18,hhtype_twoormore_family_otherfam_femalenospouse,hhtype_twoormore_family_otherfam_femalenospouse_ownchildrenunder18,hhtype_twoormore_family_otherfam_femalenospouse_noownchildrenunder18,hhtype_twoormore_nonfamily,hhtype_twoormore_nonfamily_male,hhtype_twoormore_nonfamily_female,StateFIPS,GeoFIPS
0,"Cheatham County, Tennessee",35912.0,35912.0,17981.0,1372.0,1473.0,1487.0,829.0,440.0,180.0,177.0,570.0,1133.0,1388.0,1787.0,1617.0,1411.0,1231.0,881.0,284.0,394.0,221.0,268.0,389.0,226.0,135.0,88.0,17931.0,1187.0,1330.0,1398.0,854.0,369.0,173.0,182.0,518.0,1201.0,1487.0,1780.0,1649.0,1372.0,1161.0,850.0,289.0,373.0,217.0,312.0,433.0,327.0,204.0,265.0,35912.0,35660.0,34783.0,532.0,135.0,63.0,17.0,130.0,252.0,34506.0,437.0,2.76,13508.0,13508.0,12878.0,630.0,12878.0,10773.0,2105.0,12878.0,2177.0,1042.0,1135.0,10701.0,10162.0,8356.0,4026.0,4330.0,1806.0,564.0,357.0,207.0,1242.0,714.0,528.0,539.0,330.0,209.0,47.0,21.0
1,"Davidson County, Tennessee",569891.0,569891.0,275865.0,19466.0,18200.0,17089.0,10120.0,9254.0,4730.0,4558.0,14070.0,26542.0,24511.0,24019.0,22218.0,19891.0,16467.0,11655.0,3874.0,5310.0,3010.0,4378.0,6608.0,4941.0,2961.0,1993.0,294026.0,18347.0,17524.0,16143.0,9558.0,9399.0,4935.0,4832.0,14420.0,25950.0,23184.0,24029.0,23233.0,20978.0,17698.0,13313.0,4521.0,6409.0,4016.0,5878.0,9508.0,8301.0,5841.0,6009.0,569891.0,558652.0,381783.0,147696.0,1679.0,13275.0,403.0,13816.0,11239.0,371150.0,26091.0,2.30,252977.0,252977.0,237405.0,15572.0,237405.0,131340.0,106065.0,237405.0,79249.0,34151.0,45098.0,158156.0,138106.0,94784.0,39175.0,55609.0,43322.0,9283.0,4111.0,5172.0,34039.0,19985.0,14054.0,20050.0,11527.0,8523.0,47.0,37.0
2,"Dickson County, Tennessee",43156.0,43156.0,21158.0,1538.0,1686.0,1726.0,957.0,557.0,250.0,236.0,701.0,1416.0,1594.0,1779.0,1756.0,1467.0,1361.0,1104.0,384.0,548.0,338.0,428.0,538.0,414.0,242.0,138.0,21998.0,1437.0,1584.0,1601.0,957.0,532.0,246.0,227.0,744.0,1459.0,1649.0,1753.0,1828.0,1513.0,1431.0,1129.0,381.0,556.0,322.0,505.0,693.0,579.0,462.0,410.0,43156.0,42718.0,40243.0,1978.0,172.0,116.0,5.0,204.0,438.0,40020.0,484.0,2.59,17614.0,17614.0,16473.0,1141.0,16473.0,12539.0,3934.0,16473.0,3678.0,1591.0,2087.0,12795.0,12175.0,9604.0,4322.0,5282.0,2571.0,670.0,387.0,283.0,1901.0,1162.0,739.0,620.0,384.0,236.0,47.0,43.0
3,"Houston County, Tennessee",8088.0,8088.0,3999.0,268.0,311.0,266.0,174.0,86.0,36.0,39.0,146.0,264.0,

In [28]:
data.to_csv('../../data/SF12000.csv', index = False)